# 03 - Train MLP from Pre-extracted Concerto Features

This notebook assumes Concerto features for `S3DIS Area 5` already exist in `features/s3dis_area5/`.

Workflow:
- mount Drive and sync the repo
- install project dependencies
- symlink shared Drive folders
- verify saved `.npz` feature files
- generate CLIP label embeddings if missing
- train the MLP translation head
- inspect checkpoints and history

Feature extraction belongs to `notebooks/02_feature_extraction.ipynb`, not this notebook.


### 1. Mount Drive and Get the Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline


### 2. Install Project Dependencies

In [ ]:
!pip install -q -r requirements.txt


### 3. Symlink Shared Drive Folders

In [ ]:
!rm -f data checkpoints features
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features


### 4. Verify Training Inputs

In [ ]:
from pathlib import Path
import numpy as np

features_dir = Path('features/s3dis_area5')
processed_dir = Path('data/s3dis_processed')
clip_emb_path = processed_dir / 'label_to_clip_embeddings.npy'
ckpt_dir = Path('checkpoints/mlp_s3dis')
feature_files = sorted(features_dir.glob('*.npz'))

print('features dir exists  :', features_dir.exists(), features_dir)
print('num feature files    :', len(feature_files))
print('sample files         :', [path.name for path in feature_files[:5]])
print('CLIP embeddings file :', clip_emb_path.exists(), clip_emb_path)
print('checkpoint dir exists:', ckpt_dir.exists(), ckpt_dir)

assert feature_files, 'No .npz feature files found in features/s3dis_area5. Run notebooks/02_feature_extraction.ipynb first.'

with np.load(feature_files[0]) as sample:
    sample_keys = list(sample.files)
    print('\nsample keys:', sample_keys)
    for key in sample.files:
        print(key, sample[key].shape, sample[key].dtype)

feature_key_candidates = {'features', 'feature', 'feat'}
label_key_candidates = {'labels', 'label', 'segment'}
assert any(key in sample_keys for key in feature_key_candidates), f'Missing feature array in {feature_files[0].name}.'
assert any(key in sample_keys for key in label_key_candidates), f'Missing label array in {feature_files[0].name}.'


### 5. Generate CLIP Label Embeddings if Missing

In [ ]:
from pathlib import Path
import torch
import yaml

from src.clip_utils import save_class_embeddings_numpy

config_path = Path('configs/train_mlp_s3dis.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
data_cfg = cfg['data']
clip_cfg = cfg.get('clip', {})
clip_emb_path = Path(data_cfg['label_embeddings_path'])

if clip_emb_path.exists():
    print('CLIP label embeddings already exist:', clip_emb_path)
else:
    label_texts = data_cfg.get('label_texts')
    if not label_texts:
        raise ValueError('`data.label_texts` is required to precompute label embeddings.')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    array = save_class_embeddings_numpy(
        output_path=clip_emb_path,
        class_names=label_texts,
        templates=clip_cfg.get('templates'),
        model_name=clip_cfg.get('model_name', 'ViT-B-32'),
        pretrained=clip_cfg.get('pretrained', 'openai'),
        device=device,
    )
    print(f'Saved CLIP label embeddings to {clip_emb_path} (shape: {array.shape})')


### 6. Inspect Training Config

In [ ]:
from pathlib import Path
import yaml

config_path = Path('configs/train_mlp_s3dis.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg


### 7. Train the Translation Head

In [ ]:
!python src/train.py --config configs/train_mlp_s3dis.yaml


### 8. Inspect Training Outputs

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path('checkpoints/mlp_s3dis')
print('checkpoint dir exists:', ckpt_dir.exists())
for path in sorted(ckpt_dir.glob('*')):
    print(path.name)

history_path = ckpt_dir / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    print('\nLast history rows:')
    for row in history[-5:]:
        print(row)
else:
    print('\nNo history.json found yet.')
